## Read Bronze

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

sales = spark.table(
    "e2e_project.bronze.crm_sales_details"
)

display(sales)

In [0]:
sales.printSchema()
print("Rows:", sales.count())

## Understand the relationships

Customer

   │
   │ sls_cust_id
   
   ▼
  SALE

   │
   │ sls_prd_key

   ▼
 Product


dim_customer ──┐

               │

               ▼

           fact_sales

               ▲

               │

dim_product ───┘


## Trim text columns

In [0]:
for field in sales.schema.fields:
    if isinstance(field.dataType, StringType):
        sales = sales.withColumn(
            field.name,
            trim(col(field.name))
        )

This automatically trims every string field rather than hard-coding them individually.

## Investigate the dates BEFORE changing them

In [0]:
display(
    sales.select(
        "sls_order_dt",
        "sls_ship_dt",
        "sls_due_dt"
    )
)

## Find malformed order dates

In [0]:
display(
    sales.filter(
        (col("sls_order_dt").isNull()) |
        (col("sls_order_dt") == 0) |
        (
            F.length(
                col("sls_order_dt").cast("string")
            ) != 8
        )
    )
)

This is an important Silver concept.

We're asking:

Is date NULL?

OR

Is date 0?

OR

Does it not contain 8 digits?

## Convert the dates properly

In [0]:
def clean_yyyymmdd(column_name):
    value = col(column_name).cast("string")

    return (
        F.when(
            col(column_name).isNull() |
            (col(column_name) == 0) |
            (F.length(value) != 8),
            F.lit(None).cast("date")
        )
        .otherwise(
            F.to_date(value, "yyyyMMdd")
        )
    )

In [0]:
sales = (
    sales
    .withColumn(
        "sls_order_dt",
        clean_yyyymmdd("sls_order_dt")
    )
    .withColumn(
        "sls_ship_dt",
        clean_yyyymmdd("sls_ship_dt")
    )
    .withColumn(
        "sls_due_dt",
        clean_yyyymmdd("sls_due_dt")
    )
)

This is type casting + validation, which is exactly the purpose of Silver in the project.

In [0]:
sales.printSchema()

## Investigate price problems

In [0]:
display(
    sales.filter(
        col("sls_price").isNull() |
        (col("sls_price") <= 0)
    )
)

## Correct invalid prices

In [0]:
sales = sales.withColumn(
    "sls_price",
    F.when(
        col("sls_price").isNull() |
        (col("sls_price") <= 0),

        F.when(
            col("sls_quantity") != 0,
            col("sls_sales") / col("sls_quantity")
        )
        .otherwise(None)
    )
    .otherwise(col("sls_price"))
)

##Check the sales equation

In [0]:
display(
    sales.filter(
        col("sls_sales").isNotNull() &
        col("sls_price").isNotNull() &
        col("sls_quantity").isNotNull() &
        (
            F.abs(
                col("sls_sales") -
                (col("sls_price") * col("sls_quantity"))
            ) > 0.01
        )
    )
)

## Validate date logic
A shipment generally shouldn't happen before an order.

In [0]:
display(
    sales.filter(
        col("sls_order_dt").isNotNull() &
        col("sls_ship_dt").isNotNull() &
        (
            col("sls_ship_dt") <
            col("sls_order_dt")
        )
    )
)

In [0]:
display(
    sales.filter(
        col("sls_order_dt").isNotNull() &
        col("sls_due_dt").isNotNull() &
        (
            col("sls_due_dt") <
            col("sls_order_dt")
        )
    )
)

Ideally both return no problematic records.

This is an example of a business-rule quality check rather than merely a datatype check.

## Rename the columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name, new_name in RENAME_MAP.items():
    sales = sales.withColumnRenamed(
        old_name,
        new_name
    )

Silver is becoming understandable without knowing the CRM's strange source naming convention.

## Inspect the final result

In [0]:
display(sales.limit(20))

In [0]:
sales.printSchema()

## Final quality checks

In [0]:
#check invalide price 
display(
    sales.filter(
        col("price").isNull() |
        (col("price") <= 0)
    )
)

In [0]:
#check invalid quantity
display(
    sales.filter(
        col("quantity").isNull() |
        (col("quantity") <= 0)
    )
)

In [0]:
#check identifier
display(
    sales.filter(
        col("order_number").isNull() |
        col("product_number").isNull() |
        col("customer_id").isNull()
    )
)

## Save your third Silver table

In [0]:
(
    sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "e2e_project.silver.crm_sales_details"
    )
)

In [0]:
%sql

SELECT *
FROM e2e_project.silver.crm_sales_details
LIMIT 20;

At this point CRM Silver is complete.

And the key idea is that each dataset gets the transformations appropriate to its business meaning. Silver is not simply “call dropna() on everything.